# Compare the transfer methods

Collects the `results.csv` files of the regressor stages listed in `tocompare` (family `visual_proprioception_transfer`), and writes into the data directory of the run:
- `all_results.csv`: all the rows;
- `summary.csv`: per target, method, regressor and K, the RMSE averaged over the joints, as the mean and std over the seeds;
- `summary_joints.csv`: the same per joint, also in physical units;
- `comparison_<target>.png` / `.pdf`: the mean RMSE against K for every method (with the regressors `mlp` and `ridge_meta`), and the trivial baseline `mean`.

In [ ]:
import sys
sys.path.append("../..")

from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"

import pathlib
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

matplotlib.rcParams["pdf.fonttype"] = 42

In [ ]:
# *** Initialize the variables with default values
# *** This cell should be tagged as parameters
# *** If papermill is used, some of the values will be overwritten

# If it is set to discard-old, the exprun will be recreated from scratch
creation_style = "exist-ok"

experiment = "visual_proprioception_transfer_compare"
run = "compare_example"

# If not None, set an external experiment path
expruns_path = None
# If not None, set an output path
results_path = None

In [ ]:
if expruns_path:
    Config().set_exprun_path(pathlib.Path(expruns_path))
if results_path:
    Config().set_results_path(pathlib.Path(results_path))

exp = Config().get_experiment(experiment, run, creation_style=creation_style)
results = pd.concat([
    pd.read_csv(pathlib.Path(Config().get_experiment(exp["source_experiment"], vp_run)["data_dir"], "results.csv"))
    for vp_run in exp["tocompare"]], ignore_index=True)
results.to_csv(pathlib.Path(exp["data_dir"], "all_results.csv"), index=False)

In [ ]:
# the mean over the joints for each seed, then the mean and std over the seeds
keys = ["target", "method", "regressor", "k"]
per_seed = results.groupby(keys + ["seed"])["rmse"].mean()
summary = per_seed.groupby(level=keys).agg(["mean", "std", "count"]).reset_index()
summary.to_csv(pathlib.Path(exp["data_dir"], "summary.csv"), index=False)

joints = results.groupby(keys + ["joint", "unit"])[["rmse", "rmse_physical"]].agg(["mean", "std"]).reset_index()
joints.columns = ["_".join(c for c in column if c) for column in joints.columns]
joints.to_csv(pathlib.Path(exp["data_dir"], "summary_joints.csv"), index=False)

for target, table in summary.groupby("target"):
    print(f"=== target {target}: mean RMSE (normalized) over the joints, mean over the seeds")
    print(table.pivot_table(index=["method", "regressor"], columns="k", values="mean").round(4).to_string())

In [ ]:
# the mean baseline is the same for every method (it depends only on the K
# training frames), so it is drawn once
for target, table in summary.groupby("target"):
    fig, ax = plt.subplots(figsize=(8, 5))
    curves = table[table["regressor"].isin(["mlp", "ridge_meta"])]
    for (method, regressor), curve in curves.groupby(["method", "regressor"]):
        curve = curve.sort_values("k")
        ax.errorbar(curve["k"], curve["mean"], yerr=curve["std"].fillna(0), marker="o", capsize=2,
                    label=f"{method} ({regressor})")
    baseline = table[table["regressor"] == "mean"].groupby("k")["mean"].mean()
    ax.plot(baseline.index, baseline.values, "k--", label="mean baseline")
    ax.set_xscale("log")
    ax.set_xlabel("labeled target frames (K)")
    ax.set_ylabel("RMSE (normalized), mean over joints")
    ax.set_title(f"Transfer to {target}")
    ax.legend(fontsize=7, loc="center left", bbox_to_anchor=(1.0, 0.5))
    fig.tight_layout()
    for suffix in ["png", "pdf"]:
        fig.savefig(pathlib.Path(exp["data_dir"], f"comparison_{target}.{suffix}"), bbox_inches="tight")
    plt.show()

In [ ]:
exp.done()